# Recovering Population Structure & Mapping Drug Resistance## African *Plasmodium falciparum* — Pf8 datasetThis notebook answers two questions using genome-wide genetic distance data for African malaria parasite samples:1. **Can we recover population structure using PCA/PCoA and unsupervised clustering?**2. **Do drug-resistant parasites cluster within specific genetic lineages, or are they distributed evenly across the population?****Pipeline:** load & QC-filter samples → validate & run PCoA on the genetic distance matrix → visualize structure against known labels → K-means clustering (two passes) → map clusters geographically → merge drug-resistance markers → test resistance-cluster associations → map resistance geographically.> Cleaned up from the exploratory working notebook: duplicate/dead cells, debug prints, and superseded merge attempts have been removed. Logic and results are unchanged.

## 0. ConfigurationUpdate these paths for your local environment before running.

In [ ]:
# --- file paths: update these for your environment ---SAMPLES_METADATA_PATH = "Pf8_samples.txt"                       # tab-separated sample metadataDISTANCE_MATRIX_PATH  = "Pf8_mean_genotype_distance_snp_only.npy"  # precomputed pairwise genetic distanceRESISTANCE_CSV_PATH   = "Pf8-samples.csv"                        # per-sample drug-resistance calls

## Part 1 — Recovering Population Structure

### 1.1 Load & filter data

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as sns

In [ ]:
# loading sample metadatasamples = pd.read_csv(SAMPLES_METADATA_PATH, sep="\t")samples.shape

In [ ]:
# defining africa countriesafrican_countries = [    'Mauritania',    'Gambia',    'Guinea',    'Kenya',    'Tanzania',    'Ghana',    'Burkina Faso',    'Mali',    'Malawi',    'Uganda',    'Democratic Republic of the Congo',    'Nigeria',    'Madagascar',    'Cameroon',    "Côte d'Ivoire",    'Ethiopia',    'Benin',    'Senegal',    'Gabon',    'Sudan',    'Mozambique']

In [ ]:
# filtering for africa countries that have passed qc filtered_samples = samples[    (samples["QC pass"] == True ) &    (samples["Country"].isin(african_countries))]print(filtered_samples.shape)filtered_samples.head()

In [ ]:
# loading the precomputed pairwise genetic distance matrix (memory-mapped: too large to load fully into RAM)dist = np.load(DISTANCE_MATRIX_PATH, mmap_mode="r")dist.shape

In [ ]:
# attach sample IDs to the distance matrix, then subset to our filtered African samplesids = samples["Sample"].tolist()dist_df = pd.DataFrame(dist, index=ids, columns=ids)filtered_ids = filtered_samples["Sample"].tolist()dist_africa = dist_df.loc[filtered_ids, filtered_ids]dist_africa.shape

### 1.2 Validate the distance matrixPCoA requires a symmetric matrix with an exact zero diagonal.

In [ ]:
print("NaN values:", np.isnan(dist_africa.values).sum())print("Infinite values:", np.isinf(dist_africa.values).sum())print("Symmetric:", np.allclose(dist_africa.values, dist_africa.values.T))print("Diagonal sample (should be ~0, may have float noise):", np.diag(dist_africa.values)[:5])

In [ ]:
# zero the diagonal exactly (float noise otherwise breaks PCoA's zero-diagonal requirement)dist_pcoa = dist_africa.values.copy()np.fill_diagonal(dist_pcoa, 0)print("Symmetric after fix:", np.allclose(dist_pcoa, dist_pcoa.T))

### 1.3 Run PCoAThe input is a distance matrix, not a samples × features table, so classical **Principal Coordinate Analysis (PCoA)** via `scikit-bio` is used instead of plain PCA — it is the correct ordination method for arbitrary distance matrices.

In [ ]:
# pip install scikit-bio  (uncomment if not already installed)from skbio.stats.distance import DistanceMatrixfrom skbio.stats.ordination import pcoaids_generic = [f"S{i}" for i in range(dist_pcoa.shape[0])]dm = DistanceMatrix(dist_pcoa, ids=ids_generic)pcoa_results = pcoa(dm)pcoa_results.proportion_explained.head(10)

In [ ]:
# attach real sample IDs (valid because dm.ids and filtered_samples were built in the same order),# keep the first 10 axes, and merge with sample metadatapcoa_coords = pcoa_results.samples.copy()pcoa_coords.index = filtered_samples["Sample"].valuespcoa_coords_10 = pcoa_coords.iloc[:, :10].copy()pcoa_metadata = pcoa_coords_10.merge(    filtered_samples, left_index=True, right_on="Sample")pcoa_metadata.shape

### 1.4 Visualize population structure

In [ ]:
# Overall structure, no labelsplt.figure(figsize=(8, 6))plt.scatter(pcoa_metadata["PC1"], pcoa_metadata["PC2"], s=8, alpha=0.6)plt.xlabel("PC1"); plt.ylabel("PC2")plt.title("PCoA of African Pf8 samples")plt.show()

In [ ]:
# Variance explained by each axis (scree plot)variance = pcoa_results.proportion_explained * 100plt.figure(figsize=(10, 5))plt.plot(range(1, 21), variance[:20], marker="o")plt.xticks(range(1, 21))plt.xlabel("PC axis"); plt.ylabel("Percentage variation explained (%)")plt.title("PCoA variation explained by components")plt.show()

In [ ]:
# By countryplt.figure(figsize=(10, 8))sns.scatterplot(data=pcoa_metadata, x="PC1", y="PC2", hue="Country", s=15, alpha=0.7)plt.legend(bbox_to_anchor=(1.05, 1))plt.title("Pf8 PCoA — Country structure")plt.show()

In [ ]:
# By curated Population labelplt.figure(figsize=(10, 8))sns.scatterplot(data=pcoa_metadata, x="PC1", y="PC2", hue="Population", s=15, alpha=0.7)plt.legend(bbox_to_anchor=(1.05, 1))plt.title("PCoA by Population")plt.show()

### 1.5 K-means clustering — Pass 1 (10 PCoA axes)

In [ ]:
from sklearn.preprocessing import StandardScalerX = pcoa_coords.iloc[:, :10].copy()X_scaled = StandardScaler().fit_transform(X)

In [ ]:
# Find the best number of clusters (K)from sklearn.cluster import KMeansimport matplotlib.pyplot as pltinertia = []K_range = range(2,15)for k in K_range:    km = KMeans(        n_clusters=k,        random_state=42,        n_init=10    )        km.fit(X_scaled)    inertia.append(km.inertia_)plt.figure(figsize=(8,5))plt.plot(K_range, inertia, marker="o")plt.xlabel("Number of clusters (K)")plt.ylabel("Within-cluster sum of squares")plt.title("Elbow method")plt.show()

In [ ]:
#Run K-meanskmeans = KMeans(    n_clusters=5,    random_state=42,    n_init=10)clusters = kmeans.fit_predict(X_scaled)

In [ ]:
pcoa_metadata["Cluster"] = clusters

In [ ]:
#Visualize clusters on PCoAplt.figure(figsize=(10,8))sns.scatterplot(    data=pcoa_metadata,    x="PC1",    y="PC2",    hue="Cluster",    palette="tab10",    s=15)plt.title("K-means clusters on PCoA")plt.legend()plt.show()

In [ ]:
# Compare clusters with known populationspd.crosstab(    pcoa_metadata["Cluster"],    pcoa_metadata["Population"])

### 1.6 Define Region (East / West Africa) and re-visualizeA simple, more geographically-intuitive label for validation.

In [ ]:
plot_df = pcoa_metadata.copy()

In [ ]:
east = [    "Kenya",    "Uganda",    "Tanzania",    "Malawi",    "Ethiopia"]west = [    "Nigeria",    "Ghana",    "Mali",    "Senegal",    "Gambia",    "Burkina Faso"]plot_df["Region"] = "Other"plot_df.loc[    plot_df["Country"].isin(east),    "Region"] = "East Africa"plot_df.loc[    plot_df["Country"].isin(west),    "Region"] = "West Africa"

In [ ]:
plot_df["Region"].value_counts()

In [ ]:
 #Plot PC1 vs PC2import matplotlib.pyplot as pltimport seaborn as snsplt.figure(figsize=(8,6))sns.scatterplot(    data=plot_df,    x="PC1",    y="PC2",    hue="Region",    alpha=0.7)plt.title("African Plasmodium falciparum Population Structure")plt.xlabel("PC1")plt.ylabel("PC2")plt.show()

In [ ]:
# Plot PC1 vs PC3plt.figure(figsize=(8,6))sns.scatterplot(    data=plot_df,    x="PC1",    y="PC3",    hue="Region",    alpha=0.7)plt.title("African Plasmodium falciparum Population Structure")plt.xlabel("PC1")plt.ylabel("PC3")plt.show()

In [ ]:
#Plot PC2 vs PC3plt.figure(figsize=(8,6))sns.scatterplot(    data=plot_df,    x="PC2",    y="PC3",    hue="Region",    alpha=0.7)plt.title("African Plasmodium falciparum Population Structure")plt.xlabel("PC2")plt.ylabel("PC3")plt.show()

In [ ]:
# Has the genetic structure changed over time?plt.figure(figsize=(8, 6))sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="Year", palette="viridis", s=40, alpha=0.7)plt.title("PCoA coloured by collection Year")plt.show()

### 1.7 K-means clustering — Pass 2 (PC1–PC2, k chosen rigorously)A second pass that picks k using both the elbow method **and** silhouette score, rather than a single heuristic.

In [ ]:
# Select the first two principal coordinatesX = plot_df[["PC1", "PC2"]]# Display the first few rowsX.head()

In [ ]:
from sklearn.cluster import KMeansimport matplotlib.pyplot as plt# Store inertia valuesinertia = []# Try different numbers of clustersK = range(1, 11)for k in K:    kmeans = KMeans(        n_clusters=k,        random_state=42,        n_init=10    )    kmeans.fit(X)    inertia.append(kmeans.inertia_)# Plot the elbow curveplt.figure(figsize=(8,5))plt.plot(K, inertia, marker='o')plt.xlabel("Number of Clusters (k)")plt.ylabel("Inertia (WCSS)")plt.title("Elbow Method for Choosing Optimal Number of Clusters")plt.show()

In [ ]:
from sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_scoreimport matplotlib.pyplot as plt# Store silhouette scoressilhouette_scores = []# Test k from 2 to 10K = range(2, 11)for k in K:    kmeans = KMeans(        n_clusters=k,        random_state=42,        n_init=10    )    labels = kmeans.fit_predict(X)    score = silhouette_score(X, labels)    silhouette_scores.append(score)# Plot the silhouette scoresplt.figure(figsize=(8,5))plt.plot(K, silhouette_scores, marker='o')plt.xlabel("Number of Clusters (k)")plt.ylabel("Silhouette Score")plt.title("Silhouette Analysis")plt.grid(True)plt.show()

In [ ]:
for k, score in zip(K, silhouette_scores):    print(f"k = {k}: {score:.4f}")

In [ ]:
from sklearn.cluster import KMeans# Fit K-Means with 2 clusterskmeans2 = KMeans(    n_clusters=2,    random_state=42,    n_init=10)# Assign cluster labelsplot_df["Cluster2"] = kmeans2.fit_predict(X)# View the first few assignmentsplot_df[["PC1", "PC2", "Cluster2"]].head()

In [ ]:
plt.figure(figsize=(8,6))sns.scatterplot(    data=plot_df,    x="PC1",    y="PC2",    hue="Cluster2",    palette="Set1",    s=40,    alpha=0.7)plt.title("K-Means Clustering (k = 2)")plt.show()

In [ ]:
plot_df["Cluster2"].value_counts()

In [ ]:
pd.crosstab(plot_df["Cluster2"], plot_df["Region"])

In [ ]:
pd.crosstab(plot_df["Cluster2"], plot_df["Country"])

In [ ]:
pd.crosstab(plot_df["Cluster2"], plot_df["Population"])

### 1.8 Geographic visualizationPlot clusters on a real African basemap (Natural Earth boundaries via `geopandas`).

In [ ]:
# pip install geopandas  (uncomment if not already installed)import geopandas as gpdurl = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"world = gpd.read_file(url)africa = world[world["CONTINENT"] == "Africa"]fig, ax = plt.subplots(figsize=(12, 12))africa.plot(ax=ax, color="whitesmoke", edgecolor="black")plt.title("African Countries Included in the Pf8 Dataset", fontsize=18)plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))# Africa backgroundafrica.plot(    ax=ax,    color="whitesmoke",    edgecolor="black")# Pf8 samplesax.scatter(    filtered_samples["Country longitude"],    filtered_samples["Country latitude"],    s=10,    alpha=0.5)plt.title("Pf8 African Sample Distribution", fontsize=18)plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))africa.plot(    ax=ax,    color="whitesmoke",    edgecolor="black")scatter = ax.scatter(    filtered_samples["Country longitude"],    filtered_samples["Country latitude"],    c=clusters,    s=20,    alpha=0.7,    cmap="tab10")plt.colorbar(scatter, label="KMeans Cluster")plt.title("Pf8 Genetic Clusters Across Africa", fontsize=18)plt.show()

In [ ]:
filtered_samples["Cluster"] = clusterscountry_structure = (    filtered_samples    .groupby("Country")["Cluster"]    .agg(lambda x: x.value_counts().idxmax())    .reset_index())country_structure.columns = ["Country", "Dominant_Cluster"]country_structure.head()

In [ ]:
country_mapping = {    "United Republic of Tanzania": "Tanzania",    "Democratic Republic of Congo": "Dem. Rep. Congo",    "Republic of the Congo": "Congo",    "Côte d'Ivoire": "Côte d’Ivoire"}country_structure["Map_Name"] = (    country_structure["Country"]    .replace(country_mapping))

In [ ]:
africa_cluster = africa.merge(    country_structure,    left_on="ADMIN",    right_on="Map_Name",    how="left")

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))africa_cluster.plot(    column="Dominant_Cluster",    cmap="tab10",    legend=True,    ax=ax,    edgecolor="black",    missing_kwds={        "color": "lightgrey",        "label": "No Pf8 samples"    })plt.title("Pf8 African Population Structure by Country", fontsize=18)plt.show()

### Part 1 summaryPCoA + K-means recovers a real, statistically validated West-vs-East African genetic divide:- Visual separation by Country/Population/Region, all pointing the same direction on PC1.- Unsupervised k=2 clustering recovers West-Africa-vs-rest with well under 1% misclassification against the curated Population label.- The clusters form a contiguous, geographically sensible pattern on a real map, not scattered noise.- A Year control shows no obvious temporal drift, ruling out a batch/date artifact.

## Part 2 — Drug Resistance and Population StructureDoes resistance to antimalarial drugs track the genetic clusters found in Part 1, or is it spread evenly across the population? This section merges six drug-resistance markers onto `pcoa_metadata` and tests both questions statistically and geographically.

### 2.1 Load & merge resistance data

In [ ]:
# a separate metadata file carries per-sample drug-resistance callsmetadata = pd.read_csv(RESISTANCE_CSV_PATH)metadata.head()

In [ ]:
# List of African countriesafrican_countries = [    "Algeria", "Angola", "Benin", "Botswana", "Burkina Faso",    "Burundi", "Cameroon", "Cape Verde", "Central African Republic",    "Chad", "Comoros", "Democratic Republic of the Congo",    "Djibouti", "Egypt", "Equatorial Guinea", "Eritrea",    "Eswatini", "Ethiopia", "Gabon", "Gambia", "Ghana",    "Guinea", "Guinea-Bissau", "Ivory Coast", "Kenya",    "Lesotho", "Liberia", "Libya", "Madagascar", "Malawi",    "Mali", "Mauritania", "Mauritius", "Morocco",    "Mozambique", "Namibia", "Niger", "Nigeria",    "Republic of the Congo", "Rwanda", "Senegal",    "Seychelles", "Sierra Leone", "Somalia",    "South Africa", "South Sudan", "Sudan",    "Tanzania", "Togo", "Tunisia", "Uganda",    "Zambia", "Zimbabwe"]# Filter QC passed samples from Africaresistance_africa = metadata[    (metadata["qc_pass"] == True) &    (metadata["country"].isin(african_countries))].copy()print(resistance_africa.shape)resistance_africa.head()

In [ ]:
drug_columns = [    "ARTresistant",    "CQresistant",    "MQresistant",    "PPQresistant",    "PYRresistant",    "SDXresistant",]# make sure sample ID column matches pcoa_metadata's "Sample" columnresistance_africa = resistance_africa.rename(columns={"sample_id": "Sample"})pcoa_resistance = pcoa_metadata.merge(    resistance_africa[["Sample"] + drug_columns],    on="Sample",    how="inner",)pcoa_resistance.shape

In [ ]:
pcoa_resistance[drug_columns].isna().sum()

### 2.2 A data gap worth reportingBefore testing associations: does every drug actually have a `resistant` category in the African subset?

In [ ]:
for drug in drug_columns:    print(drug)    print(pcoa_resistance[drug].value_counts(dropna=False))    print()

In [ ]:
# Artemisinin (ART) resistance: no "resistant" samples at all in the African subsetplt.figure(figsize=(10, 8))sns.scatterplot(data=pcoa_resistance, x="PC1", y="PC2", hue="ARTresistant", alpha=0.7)plt.title("Population Structure by Artemisinin Resistance")plt.show()# NOTE: MQresistant and PPQresistant show the same pattern (no "resistant" category present here) —# consistent with these resistances historically being concentrated in Southeast Asia rather than Africa.

### 2.3 Resistance frequency by genetic cluster (all 6 drugs)

In [ ]:
cluster_resistance_summary = {}for drug in drug_columns:    # remove missing resistance labels    temp = pcoa_resistance.dropna(        subset=[drug, "Cluster"]    )    # count resistance by cluster    table = pd.crosstab(        temp["Cluster"],        temp[drug]    )    # convert to percentage    percentage = (        table        .div(table.sum(axis=1), axis=0)        * 100    )    cluster_resistance_summary[drug] = percentage# View Artemisinin examplecluster_resistance_summary["ARTresistant"]

In [ ]:
for drug in drug_columns:    plt.figure(figsize=(8,5))    sns.heatmap(        cluster_resistance_summary[drug],        annot=True,        fmt=".1f",        cmap="Reds"    )    plt.title(        f"{drug}: Resistance frequency by genetic cluster"    )    plt.xlabel("Resistance category")    plt.ylabel("Cluster")    plt.show()

In [ ]:
from scipy.stats import chi2_contingencyfor drug in drug_columns:    temp = pcoa_resistance.dropna(        subset=[drug, "Cluster"]    )    table = pd.crosstab(        temp["Cluster"],        temp[drug]    )    chi2, p, dof, expected = chi2_contingency(table)    print(        drug,        "p-value:",        round(p,5)    )

**Result:** Chloroquine (CQ) resistance is dramatically concentrated in one cluster (~69% resistant vs. ~5–8% elsewhere, chi-square p ≈ 1×10⁻¹⁶). Pyrimethamine (PYR) and Sulfadoxine (SDX) resistance are present in the majority of *every* cluster — essentially fixed across the population rather than lineage-specific.

### 2.4 Multi-drug resistance burden

In [ ]:
# Find highly resistant parasitesresistant_values = [    "resistant"]pcoa_resistance["Resistance_score"] = (    pcoa_resistance[drug_columns]    .apply(        lambda row:        sum(row == "resistant"),        axis=1    ))pcoa_resistance[    "Resistance_score"].value_counts()

In [ ]:
plt.figure(figsize=(10,8))sns.scatterplot(    data=pcoa_resistance,    x="PC1",    y="PC2",    hue="Resistance_score",    palette="viridis",    s=40)plt.title(    "PCoA Colored by Multi-Drug Resistance Burden")plt.show()

### 2.5 Country-level and geographic resistance patterns

In [ ]:
# Calculate the average number of resistant drugs per countrycountry_resistance = (    pcoa_resistance    .groupby("Country")["Resistance_score"]    .mean()    .sort_values(        ascending=False    ))# Display countries ranked by resistance burdenprint(country_resistance)

In [ ]:
# Convert the series into a dataframe for plottingcountry_resistance_df = (    country_resistance    .reset_index())# Rename the resistance columncountry_resistance_df.columns = [    "Country",    "Average_resistance_score"]# Create bar plotplt.figure(figsize=(14,6))sns.barplot(    data=country_resistance_df,    x="Country",    y="Average_resistance_score")# Rotate country names for readabilityplt.xticks(    rotation=90)plt.ylabel(    "Average number of resistant drugs")plt.title(    "Multi-Drug Resistance Burden by Country")plt.tight_layout()plt.show()

In [ ]:
# Count samples belonging to each cluster in each countrycluster_country = pd.crosstab(    pcoa_resistance["Country"],    pcoa_resistance["Cluster"])# Display tablecluster_country

In [ ]:
# Convert counts into percentage within each countrycluster_country_percent = (    cluster_country    .div(        cluster_country.sum(axis=1),        axis=0    )    *100)cluster_country_percent

In [ ]:
# Plot cluster composition of each countryplt.figure(figsize=(12,7))cluster_country_percent.plot(    kind="bar",    stacked=True,    figsize=(12,7))plt.ylabel(    "Percentage of samples")plt.xlabel(    "Country")plt.title(    "Genetic Cluster Composition Across African Countries")plt.xticks(    rotation=90)plt.legend(    title="Cluster",    bbox_to_anchor=(1.05,1))plt.tight_layout()plt.show()

In [ ]:
# Calculate average resistance score by country and clustercluster_country_resistance = (    pcoa_resistance    .groupby(        [            "Country",            "Cluster"        ]    )["Resistance_score"]    .mean()    .reset_index())# Rank highest resistance combinationscluster_country_resistance = (    cluster_country_resistance    .sort_values(        "Resistance_score",        ascending=False    ))# Show top hotspotscluster_country_resistance.head(20)

In [ ]:
# Select top 20 country-cluster combinationshotspots = (    cluster_country_resistance    .head(20))plt.figure(figsize=(12,6))sns.barplot(    data=hotspots,    x="Country",    y="Resistance_score",    hue="Cluster")plt.xticks(    rotation=90)plt.ylabel(    "Average resistance score")plt.title(    "Highest Resistance Genetic Hotspots")plt.tight_layout()plt.show()

In [ ]:
# Test whether resistance score differs between countriesfrom scipy.stats import kruskalgroups = [    group["Resistance_score"].values    for name, group     in pcoa_resistance.groupby("Country")]# Kruskal-Wallis teststat, p = kruskal(    *groups)print(    "Kruskal-Wallis p-value:",    p)

### 2.6 Mapping resistance burden across Africa

In [ ]:
# Calculate average resistance score per countrycountry_map_data = (    pcoa_resistance    .groupby(        [            "Country",            "Country latitude",            "Country longitude"        ]    )    ["Resistance_score"]    .mean()    .reset_index())# View the resultcountry_map_data.head()

In [ ]:
# Merge the resistance scores with the Africa mapafrica_resistance = africa.merge(    country_resistance_df,    left_on="ADMIN",    right_on="Country",    how="left")# Check the merged dataafrica_resistance[["ADMIN", "Average_resistance_score"]].head()

In [ ]:
# Create a figure and axis for plottingfig, ax = plt.subplots(figsize=(12, 10))# Plot the average resistance score for each African countryafrica_resistance.plot(    column="Average_resistance_score",   # Column containing the resistance scores    cmap="Reds",                         # Colour map (light = low, dark = high resistance)    ax=ax,    legend=True,                         # Display colour legend    edgecolor="black",                   # Draw country borders    linewidth=0.5,    missing_kwds={        "color": "lightgrey",            # Countries with no data        "label": "No data"    })# Add a titleplt.title(    "Average Multi-Drug Resistance Burden Across Africa",    fontsize=16,    fontweight="bold")# Remove axis ticks and labelsplt.axis("off")# Display the mapplt.show()

### Part 2 summary — Answers to the nine sub-questions| # | Question | Answer ||---|----------|--------|| 1 | Confined to specific populations, or spread throughout? | Depends on drug: CQ confined to Cluster 1 (69% vs 5–8%); PYR/SDX spread throughout (majority of every cluster). || 2 | Genetically different from sensitive parasites? | Yes for CQ (χ²≈81, p≈1e-16); not meaningfully for PYR/SDX. || 3 | Which countries have the highest resistance burden? | Benin, Ethiopia, Gabon, Cameroon, DRC highest; Burkina Faso, Madagascar, Mauritania, Mali lowest. || 4 | Which populations (clusters) contain resistant parasites? | All 5 clusters carry PYR/SDX resistance; CQ resistance concentrated almost entirely in Cluster 1. || 5 | Confined to one country, or spreading across Africa? | Already spread — Cluster 1 (CQ-resistant lineage) present in 13+ countries; PYR/SDX resistance is continent-wide. || 6 | Has resistance increased over time? | **Not analyzed** — no Year × resistance comparison exists yet. Open question. || 7 | Associated with the observed population structure? | Yes for CQ (strong); weak/uninformative for PYR & SDX. || 8 | Does population structure explain the spread? | Partially — true for CQ's clonal-style spread, not for PYR/SDX's repeated independent selection. || 9 | How is resistance distributed within the population structure overall? | Heterogeneous by drug class: CQ = lineage-restricted; PYR/SDX = fixed nearly everywhere; ART/MQ/PPQ = undetected in this African subset. |

## Overall Conclusions**Part 1:** PCoA + K-means recovers real, statistically and geographically validated population structure in African *P. falciparum* — a West-vs-East African genetic divide.**Part 2:** Drug resistance is unevenly, but explainably, distributed within that structure. Chloroquine resistance is strongly lineage-specific and already spread across many countries; pyrimethamine/sulfadoxine resistance is close to fixed across the whole population regardless of lineage; artemisinin/mefloquine/piperaquine resistance is not detected in this African sample set.**Recommended next steps:**- Add a Year × resistance-marker analysis (the one open sub-question).- Compute a CQ-specific (not combined-burden) country ranking and choropleth.- Reconcile the k=5 vs. k=2 clustering choice before further downstream use of cluster labels.- Re-check low-sample-size countries (e.g. Burkina Faso) before treating their resistance scores as robust.